In [18]:
import os
import re
import pandas as pd

def leer_archivo_sismo(ruta_archivo):
    """
    Lee un archivo de sismo en formato texto y extrae
    fecha, hora y magnitud Mb (si existe).
    """
    try:
        with open(ruta_archivo, "r", encoding="latin-1") as f:
            texto = f.read()
    except Exception as e:
        print(f"Error leyendo {ruta_archivo}: {e}")
        return None

    # Fecha del sismo
    fecha_match = re.search(
        r"FECHA DEL SISMO \[GMT\]\s*:\s*(\d{4}/\d{2}/\d{2})", texto
    )

    # Hora del epicentro
    hora_match = re.search(
        r"HORA EPICENTRO \(GMT\)\s*:\s*([\d:.]+)", texto
    )

    # Magnitud Mb
    mag_match = re.search(
        r"Mb=([\d.]+)", texto
    )

    if not fecha_match or not mag_match:
        # Archivo incompleto o sin datos útiles
        return None

    return {
        "fecha": fecha_match.group(1),
        "hora": hora_match.group(1) if hora_match else None,
        "magnitud": float(mag_match.group(1))
    }


In [19]:
def cargar_sismos_carpeta(carpeta):
    """
    Recorre todos los archivos .txt de una carpeta,
    lee cada archivo de sismo y los unifica en un DataFrame.
    """
    registros = []

    for archivo in os.listdir(carpeta):
        if archivo.lower().endswith(".txt"):
            ruta = os.path.join(carpeta, archivo)
            datos = leer_archivo_sismo(ruta)

            if datos is not None:
                datos["archivo"] = archivo
                registros.append(datos)

    return pd.DataFrame(registros)


In [20]:
# Ruta de la carpeta con los archivos
carpeta = "Temblores_1964-1999"

# Cargar todos los sismos
df = cargar_sismos_carpeta(carpeta)

# Convertir fecha a datetime
df["fecha"] = pd.to_datetime(df["fecha"])

# Ordenar cronológicamente
df = df.sort_values("fecha").reset_index(drop=True)

print("Primeros registros:")
print(df.head())

print("\nTotal de registros leídos:", len(df))


Primeros registros:
       fecha      hora  magnitud       archivo
0 1964-07-06  07:22:09       6.7  CU016407.txt
1 1965-08-23  19:45:55       6.9  CU016508.txt
2 1968-02-03  05:36:12       5.6  CU016802.txt
3 1968-08-02  14:06:37       6.3  CU016808.txt
4 1970-04-29  11:22:40       5.8  PAJA7004.txt

Total de registros leídos: 2082


In [21]:
df_diario = (
    df.sort_values("magnitud", ascending=False)
      .drop_duplicates(subset=["fecha"], keep="first")
      .sort_values("fecha")
      .reset_index(drop=True)
)

print("\nTotal de sismos únicos por día:", len(df_diario))
print(df_diario.head())



Total de sismos únicos por día: 324
       fecha      hora  magnitud       archivo
0 1964-07-06  07:22:09       6.7  CU016407.txt
1 1965-08-23  19:45:55       6.9  CU016508.txt
2 1968-02-03  05:36:12       5.6  CU016802.txt
3 1968-08-02  14:06:37       6.3  CU016808.txt
4 1970-04-29  11:22:40       5.8  PAJA7004.txt


In [22]:
print("Estadísticas descriptivas de la magnitud:\n")
print(df_diario["magnitud"].describe())


Estadísticas descriptivas de la magnitud:

count    324.000000
mean       4.834568
std        0.737859
min        3.000000
25%        4.300000
50%        4.800000
75%        5.300000
max        7.000000
Name: magnitud, dtype: float64


In [23]:
df_diario["anio"] = df_diario["fecha"].dt.year

sismos_por_anio = df_diario.groupby("anio").size()

print(sismos_por_anio.head())


anio
1964    1
1965    1
1968    2
1970    1
1971    3
dtype: int64


In [24]:
magnitud_media_anual = df_diario.groupby("anio")["magnitud"].mean()

print(magnitud_media_anual.head())


anio
1964    6.70
1965    6.90
1968    5.95
1970    5.80
1971    4.80
Name: magnitud, dtype: float64


In [25]:
df_diario = df_diario.sort_values("fecha")

df_diario["magnitud_media_30d"] = (
    df_diario["magnitud"]
    .rolling(window=30, min_periods=1)
    .mean()
)


In [26]:
bins = [0, 2, 3, 4, 5, 6, 7, 8]
df_diario["rango_mag"] = pd.cut(df_diario["magnitud"], bins=bins)

frecuencia_magnitudes = df_diario["rango_mag"].value_counts().sort_index()

print(frecuencia_magnitudes)


rango_mag
(0, 2]      0
(2, 3]      1
(3, 4]     40
(4, 5]    169
(5, 6]     92
(6, 7]     22
(7, 8]      0
Name: count, dtype: int64


In [27]:
df_diario["delta_dias"] = df_diario["fecha"].diff().dt.days

print(df_diario["delta_dias"].describe())


count    323.000000
mean      40.755418
std      102.250175
min        1.000000
25%        5.000000
50%       13.000000
75%       36.000000
max      905.000000
Name: delta_dias, dtype: float64


In [28]:
# ==============================
# COMPARACIÓN DE VENTANAS
# ==============================

df_diario = df_diario.sort_values("fecha").reset_index(drop=True)

ventanas = [7, 30, 90]

for v in ventanas:
    # Media móvil
    df_diario[f"mag_media_{v}d"] = (
        df_diario["magnitud"]
        .rolling(window=v, min_periods=1)
        .mean()
    )
    
    # Máxima magnitud
    df_diario[f"mag_max_{v}d"] = (
        df_diario["magnitud"]
        .rolling(window=v, min_periods=1)
        .max()
    )
    
    # Conteo de sismos
    df_diario[f"conteo_{v}d"] = (
        df_diario["magnitud"]
        .rolling(window=v, min_periods=1)
        .count()
    )
    
    # Variabilidad
    df_diario[f"std_{v}d"] = (
        df_diario["magnitud"]
        .rolling(window=v, min_periods=1)
        .std()
    )

print(df_diario[[ 
    "mag_media_7d", "mag_media_30d", "mag_media_90d",
    "mag_max_7d", "mag_max_30d", "mag_max_90d"
]].head())


   mag_media_7d  mag_media_30d  mag_media_90d  mag_max_7d  mag_max_30d  \
0         6.700          6.700          6.700         6.7          6.7   
1         6.800          6.800          6.800         6.9          6.9   
2         6.400          6.400          6.400         6.9          6.9   
3         6.375          6.375          6.375         6.9          6.9   
4         6.260          6.260          6.260         6.9          6.9   

   mag_max_90d  
0          6.7  
1          6.9  
2          6.9  
3          6.9  
4          6.9  

In [29]:
print("Variabilidad promedio por ventana:\n")

for v in ventanas:
    print(f"{v} días:", df_diario[f"std_{v}d"].mean())


Variabilidad promedio por ventana:

7 días: 0.655038402820679
30 días: 0.691828186096185
90 días: 0.7146195830739281


In [30]:
# Target futuro simple
df_diario["mag_futura_5d"] = (
    df_diario["magnitud"]
    .shift(-5)
    .rolling(window=5)
    .max()
)

print("\nCorrelación con magnitud futura:\n")

for v in ventanas:
    corr = df_diario[f"mag_media_{v}d"].corr(df_diario["mag_futura_5d"])
    print(f"Ventana {v}d:", corr)



Correlación con magnitud futura:

Ventana 7d: 0.28720249126349023
Ventana 30d: 0.3491800863493445
Ventana 90d: 0.2935691492860148


In [31]:
import numpy as np

MAGNITUD_CORTE = 3.0
VENTANA_B = 50

def calcular_b_value(magnitudes):
    magnitudes = magnitudes[magnitudes >= MAGNITUD_CORTE]
    if len(magnitudes) < 5:
        return np.nan
    mean_mag = magnitudes.mean()
    if mean_mag == MAGNITUD_CORTE:
        return np.nan
    return np.log10(np.e) / (mean_mag - MAGNITUD_CORTE)

df_diario["b_value"] = (
    df_diario["magnitud"]
    .rolling(window=VENTANA_B)
    .apply(calcular_b_value, raw=False)
)


In [32]:
ventanas = [7, 30, 90]

print("Correlación b-value vs variables actuales:\n")

for v in ventanas:
    corr_media = df_diario["b_value"].corr(df_diario[f"mag_media_{v}d"])
    corr_max = df_diario["b_value"].corr(df_diario[f"mag_max_{v}d"])
    
    print(f"Ventana {v}d → media: {corr_media:.3f}, max: {corr_max:.3f}")


Correlación b-value vs variables actuales:

Ventana 7d → media: -0.349, max: -0.304
Ventana 30d → media: -0.843, max: -0.868
Ventana 90d → media: -0.826, max: -0.685


In [33]:
print("\nCorrelación con magnitud futura 5d:\n")

corr_b = df_diario["b_value"].corr(df_diario["mag_futura_5d"])
print("b-value:", corr_b)

for v in ventanas:
    corr_v = df_diario[f"mag_media_{v}d"].corr(df_diario["mag_futura_5d"])
    print(f"Ventana {v}d:", corr_v)



Correlación con magnitud futura 5d:

b-value: -0.09873263706392947
Ventana 7d: 0.28720249126349023
Ventana 30d: 0.3491800863493445
Ventana 90d: 0.2935691492860148
